In [2]:
import pandas as pd
from pathlib import Path

# 실행 위치가 프로젝트 폴더든 raw 폴더든 작동하도록 설정
DATA_DIR = Path("raw") if Path("raw").exists() else Path(".")

portfolio = pd.read_json(DATA_DIR / "portfolio.json", lines=True)
profile = pd.read_json(DATA_DIR / "profile.json", lines=True)
transcript = pd.read_json(DATA_DIR / "transcript.json", lines=True)

print("portfolio:", portfolio.shape)
print("profile:", profile.shape)
print("transcript:", transcript.shape)

portfolio: (10, 6)
profile: (17000, 5)
transcript: (306534, 4)


In [3]:
display(portfolio.head())
display(profile.head())
display(transcript.head())

print("\n[portfolio 정보]")
portfolio.info()

print("\n[profile 정보]")
profile.info()

print("\n[transcript 정보]")
transcript.info()

,reward,channels,difficulty,duration,offer_type,id
0,10,"[email, mobile, social]",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"[web, email, mobile, social]",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"[web, email, mobile]",0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,"[web, email, mobile]",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,"[web, email]",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7


,gender,age,id,became_member_on,income
0,NaN,118,68be06ca386d4c31939f3a4f0e3dd783,20170212,NaN
1,F,55,0610b486422d4921ae7d2bf64640c50b,20170715,112000.0
2,NaN,118,38fe809add3b4fcf9315a9694bb96ff5,20180712,NaN
3,F,75,78afa995795e4d85b5d9ceeca43f5fef,20170509,100000.0
4,NaN,118,a03223e636434f42ac4c3df47e8bac43,20170804,NaN


,person,event,value,time
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0



[portfolio 정보]
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   reward      10 non-null     int64 
 1   channels    10 non-null     object
 2   difficulty  10 non-null     int64 
 3   duration    10 non-null     int64 
 4   offer_type  10 non-null     str   
 5   id          10 non-null     str   
dtypes: int64(3), object(1), str(2)
memory usage: 612.0+ bytes

[profile 정보]
<class 'pandas.DataFrame'>
RangeIndex: 17000 entries, 0 to 16999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            14825 non-null  str    
 1   age               17000 non-null  int64  
 2   id                17000 non-null  str    
 3   became_member_on  17000 non-null  int64  
 4   income            14825 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 664.2 KB

[transcrip

## 1. 데이터 기본 구조 확인

### 확인 목적

본격적인 분석에 앞서 각 데이터셋의 크기, 컬럼명, 데이터 형식과 결측치 여부를 확인했다.

---

### 1) portfolio: 프로모션 정보

- 데이터 크기: 10행 × 6열
- 프로모션 10개의 조건과 발송 방식을 담고 있다.
- 현재 확인된 결측치는 없다.

| 컬럼명 | 의미 |
|---|---|
| reward | 프로모션 완료 시 제공되는 보상 |
| channels | 프로모션 발송 채널 |
| difficulty | 보상을 받기 위한 최소 결제금액 |
| duration | 프로모션 유효기간 |
| offer_type | 프로모션 유형 |
| id | 프로모션 고유번호 |

#### 프로모션 유형

- bogo: 하나를 구매하면 하나를 추가로 제공하는 프로모션
- discount: 일정 금액 이상 결제하면 할인 또는 보상을 제공하는 프로모션
- informational: 별도의 보상 없이 상품이나 혜택을 안내하는 프로모션

---

### 2) profile: 고객 정보

- 데이터 크기: 17,000행 × 5열
- 고객의 인구통계 정보와 가입일을 담고 있다.
- 일부 고객의 성별과 소득 정보가 누락되어 있다.
- 나이가 118세로 입력된 경우는 실제 나이라기보다 고객정보 미입력을 나타내는 값일 가능성이 있다.

| 컬럼명 | 의미 |
|---|---|
| gender | 고객 성별 |
| age | 고객 나이 |
| id | 고객 고유번호 |
| became_member_on | 회원 가입일 |
| income | 고객 연소득 |

`became_member_on`은 숫자로 저장되어 있지만 실제 의미는 날짜이므로 이후 날짜 형식으로 변환할 필요가 있다.

---

### 3) transcript: 고객 행동 기록

- 데이터 크기: 306,534행 × 4열
- 고객의 프로모션 수신·열람·완료 및 결제 행동을 담고 있다.

| 컬럼명 | 의미 |
|---|---|
| person | 행동을 수행한 고객의 고유번호 |
| event | 발생한 행동의 종류 |
| value | 프로모션 고유번호 또는 결제금액 |
| time | 실험 시작 후 행동이 발생하기까지 지난 시간 |

`value`에는 프로모션 고유번호나 결제금액이 딕셔너리 형태로 들어 있으므로 이후 별도의 컬럼으로 분리할 필요가 있다.

---

### 데이터 연결 기준

- `profile`의 `id`와 `transcript`의 `person`을 이용해 고객 정보와 행동 기록을 연결할 수 있다.
- `portfolio`의 `id`와 `transcript`의 `value` 안에 있는 프로모션 ID를 이용해 프로모션 정보와 행동 기록을 연결할 수 있다.

---

### 확인 결과

세 데이터셋은 모두 정상적으로 불러와졌다.  
분석 전 고객정보의 결측치와 나이 118세 이상치를 처리하고, 가입일을 날짜 형식으로 변환하며, `value` 안의 정보를 별도 컬럼으로 분리해야 한다.

In [6]:
# 데이터별 결측치와 중복 행 확인
for name, df in {
    "portfolio": portfolio,
    "profile": profile,
    "transcript": transcript
}.items():
    print(f"\n[{name}]")
    print("전체 행 수:", len(df))

    # 리스트와 딕셔너리를 문자열로 바꾼 뒤 중복 확인
    duplicate_count = df.astype(str).duplicated().sum()
    print("중복 행 수:", duplicate_count)

    print("\n컬럼별 결측치:")
    display(df.isnull().sum().to_frame("결측치 수"))

# 고객 데이터의 이상치 확인
print("\n[profile 이상치 확인]")
print("고객 ID 중복:", profile["id"].duplicated().sum())
print("118세 고객 수:", (profile["age"] == 118).sum())
print("최소 나이:", profile["age"].min())
print("최대 나이:", profile["age"].max())

print("\n성별 분포:")
display(profile["gender"].value_counts(dropna=False))

print("\n소득 요약:")
display(profile["income"].describe())



[portfolio]
전체 행 수: 10
중복 행 수: 0

컬럼별 결측치:


,결측치 수
reward,0
channels,0
difficulty,0
duration,0
offer_type,0
id,0



[profile]
전체 행 수: 17000
중복 행 수: 0

컬럼별 결측치:


,결측치 수
gender,2175
age,0
id,0
became_member_on,0
income,2175



[transcript]
전체 행 수: 306534
중복 행 수: 397

컬럼별 결측치:


,결측치 수
person,0
event,0
value,0
time,0



[profile 이상치 확인]
고객 ID 중복: 0
118세 고객 수: 2175
최소 나이: 18
최대 나이: 118

성별 분포:


gender
M      8484
F      6129
NaN    2175
O       212
Name: count, dtype: int64


소득 요약:


count     14825.000000
mean      65404.991568
std       21598.299410
min       30000.000000
25%       49000.000000
50%       64000.000000
75%       80000.000000
max      120000.000000
Name: income, dtype: float64

## 2. 결측치·중복값·이상치 점검

### 확인 목적

분석 결과가 왜곡되는 것을 방지하기 위해 데이터별 결측치, 중복 행과 비정상적인 값을 확인했다.

### 점검 결과

#### portfolio

- 전체 10행
- 중복 행 없음
- 모든 컬럼에 결측치 없음
- 프로모션 정보는 별도의 결측치 처리 없이 사용할 수 있다.

#### profile

- 전체 고객 17,000명
- 중복 행과 고객 ID 중복 없음
- 성별과 소득이 각각 2,175명에게서 누락됨
- 나이가 118세인 고객도 2,175명으로 확인됨
- 성별·소득 결측치와 118세가 같은 고객에게 나타나는지 추가 확인이 필요함
- 실제 고객 나이의 최솟값은 18세이며, 118세는 정보 미입력 표시값일 가능성이 높음
- 소득은 30,000~120,000 범위로 특별한 이상치가 확인되지 않음

#### transcript

- 전체 행동 기록 306,534건
- 컬럼별 결측치 없음
- 동일하게 보이는 기록 397건 확인
- 동일 시간에 같은 행동이 실제로 발생했을 가능성이 있으므로 중복 기록을 바로 삭제하지 않고 세부 내용을 확인해야 함

### 정리

프로모션 정보는 결측치나 중복 없이 정상적이다. 고객 데이터에서는 정보 미입력 고객에 대한 처리가 필요하며, 행동 데이터에서는 중복처럼 보이는 397건이 실제 오류인지 추가 확인해야 한다.

In [7]:
print("[행동 유형별 기록 수]")
display(transcript["event"].value_counts())

print("\n[행동 유형별 value 예시]")
for event in transcript["event"].unique():
    print(f"\n{event}")
    display(transcript.loc[transcript["event"] == event, "value"].head())

print("\n[시간 범위]")
print("최소 시간:", transcript["time"].min())
print("최대 시간:", transcript["time"].max())
print("고유 시간 수:", transcript["time"].nunique())

[행동 유형별 기록 수]


event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33579
Name: count, dtype: int64


[행동 유형별 value 예시]

offer received


0    {'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'}
1    {'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}
2    {'offer id': '2906b810c7d4411798c6938adc9daaa5'}
3    {'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'}
4    {'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'}
Name: value, dtype: object


offer viewed


...


transaction


12654    {'amount': 0.8300000000000001}
12657                 {'amount': 34.56}
12659                 {'amount': 13.23}
12670                 {'amount': 19.51}
12671                 {'amount': 18.97}
Name: value, dtype: object


offer completed


12658    {'offer_id': '2906b810c7d4411798c6938adc9daaa5...
12672    {'offer_id': 'fafdcd668e3743c1bb461111dcafc2a4...
12679    {'offer_id': '9b98b8c7a33c4b65b9aebfe6a799e6d9...
12692    {'offer_id': 'ae264e3637204a6fb9bb56bc8210ddfd...
12697    {'offer_id': '4d5c57ea9a6940dd891ad53e9dbe8da0...
Name: value, dtype: object


[시간 범위]
최소 시간: 0
최대 시간: 714
고유 시간 수: 120


## 3. 고객 행동 기록 구조 확인

### 확인 목적

고객 행동의 종류와 행동별 저장 형식을 확인하고, 프로모션 성과를 측정하기 위해 필요한 전처리 항목을 파악했다.

### 행동 유형

| 행동 | 기록 수 | 의미 |
|---|---:|---|
| transaction | 138,953 | 고객의 결제 |
| offer received | 76,277 | 프로모션 수신 |
| offer viewed | 57,725 | 프로모션 열람 |
| offer completed | 33,579 | 프로모션 조건 달성 |

### value 컬럼의 구조

- 프로모션 수신과 열람 기록에는 `offer id`가 저장되어 있다.
- 프로모션 완료 기록에는 `offer_id`와 보상금액이 저장되어 있다.
- 결제 기록에는 결제금액인 `amount`가 저장되어 있다.
- 같은 프로모션 ID가 `offer id`와 `offer_id`라는 서로 다른 이름으로 저장되어 있으므로 하나의 컬럼으로 통일해야 한다.

### 시간 범위

- 최소 시간: 0
- 최대 시간: 714
- 고유 시간: 120개
- `time`은 실제 날짜가 아니라 실험 시작 후 경과 시간을 의미한다.

### 해석 시 주의점

전체 수신·열람·완료 기록 수만 나누어 전환율을 계산하면 안 된다. 한 고객이 여러 프로모션을 받을 수 있고, 프로모션마다 유형과 유효기간이 다르기 때문이다. 특히 정보 제공형 프로모션은 완료 이벤트가 발생하지 않는다.

따라서 고객 ID, 프로모션 ID와 발생 시간을 연결한 후 프로모션별 수신·열람·완료 과정을 구성해야 한다.

In [8]:
# 1. 고객정보 미입력 패턴 확인
missing_pattern = profile.assign(
    gender_missing=profile["gender"].isna(),
    income_missing=profile["income"].isna(),
    age_is_118=profile["age"].eq(118)
)[["gender_missing", "income_missing", "age_is_118"]]

print("[고객정보 미입력 패턴]")
display(missing_pattern.value_counts().to_frame("고객 수"))


# 2. 실제 연령 범위 확인
valid_age = profile.loc[profile["age"] != 118, "age"]

print("\n[118세 제외 연령 요약]")
display(valid_age.describe())


# 3. 가입일 형식과 기간 확인
member_date = pd.to_datetime(
    profile["became_member_on"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

print("\n[가입일 확인]")
print("날짜 변환 실패:", member_date.isna().sum())
print("가장 이른 가입일:", member_date.min())
print("가장 늦은 가입일:", member_date.max())


# 4. profile profile과 transcript의 고객 ID 연결 여부 확인
profile_ids = set(profile["id"])
transcript_person_ids = set(transcript["person"])

print("\n[고객 ID 연결 확인]")
print("profile에 없는 행동 고객 수:",
      len(transcript_person_ids - profile_ids))
print("행동 기록이 없는 profile 고객 수:",
      len(profile_ids - transcript_person_ids))


# 5. transcript 안의 프로모션 ID를 임시로 추출
# 원본 데이터에는 새로운 컬럼을 만들지 않음
transcript_offer_ids = transcript["value"].apply(
    lambda x: x.get("offer id", x.get("offer_id"))
)

recorded_offer_ids = set(transcript_offer_ids.dropna())
portfolio_offer_ids = set(portfolio["id"])

print("\n[프로모션 ID 연결 확인]")
print("portfolio에 없는 프로모션 ID 수:",
      len(recorded_offer_ids - portfolio_offer_ids))
print("행동 기록이 없는 프로모션 ID 수:",
      len(portfolio_offer_ids - recorded_offer_ids))


# 6. 프로모션 기본 구성 확인
print("\n[프로모션 유형별 개수]")
display(portfolio["offer_type"].value_counts())

print("\n[프로모션 수치형 컬럼 요약]")
display(portfolio[["reward", "difficulty", "duration"]].describe())

[고객정보 미입력 패턴]


,,,고객 수
gender_missing,income_missing,age_is_118,
False,False,False,14825
True,True,True,2175



[118세 제외 연령 요약]


count    14825.000000
mean        54.393524
std         17.383705
min         18.000000
25%         42.000000
50%         55.000000
75%         66.000000
max        101.000000
Name: age, dtype: float64


[가입일 확인]
날짜 변환 실패: 0
가장 이른 가입일: 2013-07-29 00:00:00
가장 늦은 가입일: 2018-07-26 00:00:00

[고객 ID 연결 확인]
profile에 없는 행동 고객 수: 0
행동 기록이 없는 profile 고객 수: 0

[프로모션 ID 연결 확인]
portfolio에 없는 프로모션 ID 수: 0
행동 기록이 없는 프로모션 ID 수: 0

[프로모션 유형별 개수]


offer_type
bogo             4
discount         4
informational    2
Name: count, dtype: int64


[프로모션 수치형 컬럼 요약]


,reward,difficulty,duration
count,10.000000,10.000000,10.000000
mean,4.200000,7.700000,6.500000
std,3.583915,5.831905,2.321398
min,0.000000,0.000000,3.000000
25%,2.000000,5.000000,5.000000
50%,4.000000,8.500000,7.000000
75%,5.000000,10.000000,7.000000
max,10.000000,20.000000,10.000000


## 4. 데이터 연결 및 범위 확인

### 확인 목적

고객정보의 결측 패턴을 확인하고, 고객 및 프로모션 정보가 행동 기록과 정상적으로 연결되는지 점검했다. 또한 고객 연령, 가입일과 프로모션 조건의 범위를 확인했다.

### 고객정보 미입력 패턴

- 정상적으로 고객정보가 입력된 고객은 14,825명이다.
- 성별과 소득이 누락되고 나이가 118세로 기록된 고객은 2,175명이다.
- 세 가지 조건이 동일한 고객에게 함께 나타나므로, 118세는 실제 나이가 아니라 고객정보 미입력을 나타내는 표시값으로 판단된다.
- 이 고객들은 전체 행동 분석에는 포함할 수 있지만, 연령·성별·소득별 분석에서는 별도 처리해야 한다.

### 고객 연령

- 118세를 제외한 실제 연령 범위는 18~101세이다.
- 평균 연령은 약 54.4세, 중앙값은 55세이다.
- 최대 나이 101세는 높은 값이지만 현실적으로 불가능하다고 단정할 수 없어 유지한다.

### 회원 가입일

- 모든 가입일이 날짜 형식으로 정상 변환되었다.
- 최초 가입일은 2013년 7월 29일이다.
- 가장 최근 가입일은 2018년 7월 26일이다.
- 이후 가입 연도나 가입 기간을 파생변수로 만들 수 있다.

### 데이터 연결 결과

- `profile.id`와 `transcript.person`에 연결되지 않는 고객은 없다.
- `portfolio.id`와 행동 기록의 프로모션 ID에 연결되지 않는 프로모션은 없다.
- 따라서 고객 정보, 프로모션 정보와 행동 기록을 모두 연결할 수 있다.

### 프로모션 구성

- BOGO 프로모션: 4개
- 할인형 프로모션: 4개
- 정보 제공형 프로모션: 2개
- 보상 범위: 0~10
- 달성 조건 범위: 0~20
- 유효기간 범위: 3~10일

정보 제공형 프로모션은 보상과 달성 조건이 없으며 완료 이벤트가 발생하지 않는다. 따라서 BOGO·할인형과 동일한 완료율 기준으로 평가해서는 안 된다.

### 최종 점검 결과

세 데이터셋은 고객 ID와 프로모션 ID를 기준으로 모두 정상적으로 연결할 수 있다. 분석 전 고객정보 미입력값, 날짜 형식, 중첩된 `value` 컬럼과 프로모션 ID 표기를 정리해야 한다.

## 데이터 점검 최종 정리 및 팀 논의 사항

### 1. 데이터 구성

| 데이터 | 크기 | 주요 내용 |
|---|---:|---|
| portfolio | 10행 × 6열 | 프로모션 유형, 보상, 달성 조건, 기간, 발송 채널 |
| profile | 17,000행 × 5열 | 고객 성별, 나이, 소득, 가입일 |
| transcript | 306,534행 × 4열 | 프로모션 수신·열람·완료 및 결제 기록 |

세 데이터셋은 고객 ID와 프로모션 ID를 기준으로 모두 연결할 수 있으며, 연결되지 않는 고객이나 프로모션은 없다.

---

### 2. 추후 통합·변환이 필요한 항목

#### 고객 ID 통합

- `profile.id`와 `transcript.person`은 같은 고객 고유번호이다.
- 데이터 통합 후에는 `customer_id`와 같은 하나의 컬럼명으로 통일할 수 있다.
- 원본 컬럼은 유지하고 분석용 데이터에서 통일하는 것이 안전하다.

#### 프로모션 ID 통합

- `portfolio.id`는 프로모션 고유번호이다.
- `transcript.value`에서는 행동에 따라 `offer id` 또는 `offer_id`로 다르게 기록되어 있다.
- 두 값은 같은 역할을 하므로 분석용 데이터에서는 `offer_id`로 통일할 필요가 있다.

#### value 컬럼 분리

`transcript.value`에는 행동에 따라 서로 다른 정보가 들어 있다.

- 수신·열람: 프로모션 ID
- 완료: 프로모션 ID와 보상금액
- 결제: 결제금액

따라서 분석용 데이터에서는 다음과 같이 분리할 필요가 있다.

- `offer_id`: 프로모션 고유번호
- `amount`: 결제금액
- `reward_received`: 실제 지급된 보상금액

#### 날짜와 시간 변환

- `became_member_on`은 숫자로 저장되어 있지만 실제로는 가입일이므로 날짜 형식으로 변환해야 한다.
- 가입일을 이용해 가입 연도 또는 가입 기간을 만들 수 있다.
- `transcript.time`은 실제 날짜가 아니라 실험 시작 후 경과 시간이며 단위는 시간이다.
- 필요하면 `time ÷ 24`를 이용해 경과 일수를 만들 수 있다.

#### 발송 채널 처리

- `portfolio.channels`에는 여러 채널이 하나의 리스트로 저장되어 있다.
- 채널별 성과를 분석하려면 web, email, mobile, social을 각각 별도 컬럼으로 만들거나 행으로 분리해야 한다.

---

### 3. 데이터 품질 관련 확인 사항

#### 고객정보 미입력

- 성별과 소득이 누락되고 나이가 118세로 기록된 고객이 2,175명이다.
- 세 조건이 같은 고객에게 동시에 나타나므로 118세는 고객정보 미입력을 나타내는 표시값으로 판단된다.
- 118세를 제외한 실제 고객 연령 범위는 18~101세이다.

#### 중복 기록

- portfolio와 profile에는 중복 행이 없다.
- transcript에는 동일하게 보이는 기록이 397건 존재한다.
- 같은 고객이 같은 시간에 동일한 행동을 실제로 했을 가능성이 있으므로 바로 삭제하지 않고 이벤트별 세부 내용을 확인해야 한다.

#### 데이터 연결

- `profile.id`와 `transcript.person`은 모두 정상적으로 연결된다.
- `portfolio.id`와 행동 기록의 프로모션 ID도 모두 정상적으로 연결된다.

---

### 4. 팀에서 결정해야 할 사항

#### 고객정보 미입력 처리

- 해당 고객을 분석에서 제외할지
- 전체 성과 분석에는 포함하고 고객 특성별 분석에서만 제외할지
- 성별·연령대·소득대를 `Unknown`으로 분류할지 결정해야 한다.

#### 중복 기록 처리

- transcript의 중복처럼 보이는 397건이 실제 중복인지 확인해야 한다.
- 실제 데이터 오류로 판단되는 경우에만 제거한다.

#### 고객 구간 설정

다음 항목의 구간 기준을 팀에서 통일해야 한다.

- 연령대
- 소득대
- 가입 기간
- 결제금액대

#### 프로모션 성과 기준

- 수신 대비 열람률
- 수신 대비 완료율
- 열람 후 완료율
- 프로모션별 결제금액
- 고객당 결제금액

어떤 지표를 핵심 성과로 사용할지 정해야 한다.

#### 정보 제공형 프로모션 평가

- 정보 제공형 프로모션은 보상과 달성 조건이 없어 완료 이벤트가 발생하지 않는다.
- 따라서 BOGO·할인형과 같은 완료율로 비교할 수 없다.
- 정보 제공형은 열람률이나 프로모션 노출 이후 결제 변화 등 별도의 기준으로 평가해야 한다.

#### 결제와 프로모션의 연결 기준

- 결제 기록에는 프로모션 ID가 직접 들어 있지 않다.
- 특정 결제가 어떤 프로모션의 영향으로 발생했는지 판단하려면 고객 ID, 수신·열람 시점, 프로모션 유효기간을 이용한 연결 기준이 필요하다.
- 프로모션을 받기 전의 결제나 유효기간이 지난 후의 결제를 성과에 포함할지 팀에서 명확히 정해야 한다.

---

### 5. 전처리 원칙

원본 데이터는 수정하지 않고 별도의 분석용 데이터를 생성한다. 분석용 데이터에서는 고객 ID와 프로모션 ID의 컬럼명을 통일하고, 중첩된 `value`를 분리하며, 날짜·시간과 고객 구간을 분석 목적에 맞게 변환한다.